# 04 真实 COF 数据：从数据库到可建模数据集

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanteen/COF-ML-Tutorial/blob/main/notebooks/04_real_cof_dataset.ipynb)

## Learning objectives
直接读取 `Wanteen/CURATED-COFs`，完成 schema inspection、missing values、duplicates、metadata/feature 区分，并设计避免近重复泄漏的数据划分。

In [ ]:
import pandas as pd
url='https://raw.githubusercontent.com/Wanteen/CURATED-COFs/master/cof-frameworks.csv'
df=pd.read_csv(url)
print('shape =',df.shape)
display(df.head())
print('Columns:',df.columns.tolist())

## 1. Data audit 先于 model.fit()
拿到数据库后不要马上训练。先回答：每一行是什么？每一列是什么？单位是什么？缺失值代表没有测量还是物理上的 0？同一个 COF 是否有多个名字/多个结构版本？

In [ ]:
summary=pd.DataFrame({
 'dtype':df.dtypes.astype(str),
 'missing':df.isna().sum(),
 'missing_%':100*df.isna().mean(),
 'unique':df.nunique(dropna=True)
})
display(summary.sort_values('missing_%',ascending=False).head(30))

In [ ]:
num=df.select_dtypes(include='number')
print('Numeric columns:',num.columns.tolist())
display(num.describe().T)

## 2. COF 数据中的常见 leakage
- 同一结构的别名/重复记录跨 train/test；
- 同一 building block family 的高度相似结构随机分散；
- 用目标性质计算得到的派生量反过来预测目标；
- 同一论文/同一计算 workflow 的系统性偏差同时出现在 train/test。

因此 random split 只能作为第一条 baseline，不能自动等价于“未知 COF 泛化能力”。

## 3. Metadata ≠ feature
论文 DOI、数据库 ID、作者、文件名通常用于追踪 provenance，而不是直接作为物理 feature。模型输入应该有明确的材料含义。保留 metadata 很重要，但不要因为它是字符串/数字就自动喂给模型。

## Exercises
1. 找出缺失率最高的 10 列。
2. 找出可能的 categorical variables。
3. 把字段分为 `identifier / provenance / feature candidate / target candidate`。
4. 检查可能的重复记录。
5. 设计一个 family-aware split，并写出它比 random split 更严格的原因。

### Take-home message
真实材料 ML 项目通常先花大量时间理解数据；一个错误的数据集可以让任何先进模型得到没有科研意义的高分。